# All-22 Player Detection — YOLO Fine-Tuning
Run this notebook in Google Colab. Each session trains a block of epochs and saves a checkpoint to Google Drive so you can resume across sessions.

**First time:** uploads data from Drive, starts training from scratch.  
**Subsequent sessions:** detects existing checkpoint and resumes automatically.

## Step 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# All training files live here in your Drive
DRIVE_DIR = '/content/drive/MyDrive/all22_training'
os.makedirs(f'{DRIVE_DIR}/images', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/labels', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/checkpoints', exist_ok=True)

print('Drive mounted.')
print(f'Images in Drive: {len(os.listdir(DRIVE_DIR + "/images"))}')
print(f'Labels in Drive: {len(os.listdir(DRIVE_DIR + "/labels"))}')
checkpoint = f'{DRIVE_DIR}/checkpoints/last.pt'
print(f'Checkpoint exists: {os.path.exists(checkpoint)}')

## Step 2 — Install dependencies

In [ ]:
!pip install ultralytics -q
import torch
print(f'PyTorch: {torch.__version__}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NOT AVAILABLE — change runtime to GPU!"}')

## Step 3 — Set up local workspace and data

In [ ]:
import shutil, random

WORK_DIR = '/content/all22'
TRAIN_IMG = f'{WORK_DIR}/train/images'
TRAIN_LBL = f'{WORK_DIR}/train/labels'
VAL_IMG   = f'{WORK_DIR}/val/images'
VAL_LBL   = f'{WORK_DIR}/val/labels'

for d in [TRAIN_IMG, TRAIN_LBL, VAL_IMG, VAL_LBL]:
    os.makedirs(d, exist_ok=True)

# Get all images that have a matching label
all_images = sorted([
    f for f in os.listdir(f'{DRIVE_DIR}/images')
    if f.endswith('.jpg') and
    os.path.exists(f'{DRIVE_DIR}/labels/{f.replace(".jpg", ".txt")}')
])
print(f'Found {len(all_images)} labeled images')

# Reproducible 85/15 train/val split
random.seed(42)
random.shuffle(all_images)
val_count = max(1, int(len(all_images) * 0.15))
val_set   = set(all_images[:val_count])
train_set = set(all_images[val_count:])
print(f'Split: {len(train_set)} train, {len(val_set)} val')

# Copy to local workspace (faster I/O during training)
for fname in all_images:
    lname = fname.replace('.jpg', '.txt')
    if fname in val_set:
        shutil.copy(f'{DRIVE_DIR}/images/{fname}', f'{VAL_IMG}/{fname}')
        shutil.copy(f'{DRIVE_DIR}/labels/{lname}', f'{VAL_LBL}/{lname}')
    else:
        shutil.copy(f'{DRIVE_DIR}/images/{fname}', f'{TRAIN_IMG}/{fname}')
        shutil.copy(f'{DRIVE_DIR}/labels/{lname}', f'{TRAIN_LBL}/{lname}')

# Write dataset.yaml
yaml_path = f'{WORK_DIR}/dataset.yaml'
with open(yaml_path, 'w') as f:
    f.write(f'path: {WORK_DIR}\n')
    f.write('train: train/images\n')
    f.write('val: val/images\n\n')
    f.write('nc: 1\n')
    f.write('names:\n')
    f.write('  0: player\n')

print('Workspace ready.')

## Step 4 — Train

**Settings to adjust each session:**
- `EPOCHS_THIS_SESSION`: how many epochs to run this session (20-25 is safe for free tier)
- `TOTAL_EPOCHS`: target total across all sessions (50)

The cell auto-detects whether a checkpoint exists and resumes from it.

In [ ]:
import os
from ultralytics import YOLO

EPOCHS_THIS_SESSION = 20   # <-- adjust per session
TOTAL_EPOCHS        = 60   # <-- total target (keep consistent)

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

checkpoint = f'{DRIVE_DIR}/checkpoints/last.pt'
base_model = 'yolo12x.pt'  # auto-downloaded by ultralytics if not present

if os.path.exists(checkpoint):
    print(f'Resuming from checkpoint: {checkpoint}')
    # Copy checkpoint locally for faster access
    shutil.copy(checkpoint, '/content/last.pt')
    model = YOLO('/content/last.pt')
    resume = True
else:
    print(f'No checkpoint found — starting from {base_model}')
    model = YOLO(base_model)
    resume = False

model.train(
    data=yaml_path,
    epochs=TOTAL_EPOCHS,
    # On resume, YOLO uses the epoch count from the checkpoint
    # and continues until TOTAL_EPOCHS is reached
    resume=resume,
    batch=4,
    imgsz=1280,
    device='cuda',
    project='/content/runs',
    name='all22',
    exist_ok=True,
    freeze=10,           # freeze backbone, train detection head only
    # Augmentation
    hsv_h=0.01,
    hsv_s=0.3,
    hsv_v=0.2,
    degrees=0,
    translate=0.1,
    scale=0.3,
    fliplr=0.5,
    flipud=0.0,
    mosaic=0.5,
)

print('Training block complete.')

## Step 5 — Save checkpoints to Drive
**Always run this cell before closing the session.**

In [ ]:
weights_dir = '/content/runs/all22/weights'

for fname in ['last.pt', 'best.pt']:
    src = f'{weights_dir}/{fname}'
    dst = f'{DRIVE_DIR}/checkpoints/{fname}'
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f'Saved {fname} to Drive')
    else:
        print(f'WARNING: {fname} not found at {src}')

# Also save results CSV so you can track progress across sessions
results_src = '/content/runs/all22/results.csv'
if os.path.exists(results_src):
    shutil.copy(results_src, f'{DRIVE_DIR}/checkpoints/results.csv')
    print('Saved results.csv to Drive')

print('\nAll checkpoints saved. Safe to close session.')

## Step 6 — (Final session only) Download best.pt
Run this after your last training session to download the final model.

In [ ]:
from google.colab import files
files.download(f'{DRIVE_DIR}/checkpoints/best.pt')